# risk-management_004

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (232).ipynb`

| Field | Value |
|---|---|
| Section | `risk_management` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 2 |
| Detected functions | header, add_section, add_image |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
!pip install fpdf

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from fpdf import FPDF

# === LOAD DATA ===
df = pd.read_csv("/content/drive/MyDrive/tradebook_CRYPTO (3).csv")
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
df['Exit Time'] = pd.to_datetime(df['Exit Time'])

# === PARAMETERS ===
capital_per_trade = 100000
fee_perc = 0.0004  # 0.04% per side
slip_perc = 0.0005  # 0.05% per side
tp_perc = 5.0       # Take Profit 5%
sl_perc = 0.1       # Stop Loss 0.1%

# === PREP DATA ===
df['Date'] = df['Entry Time'].dt.date
df['Month'] = df['Exit Time'].dt.to_period('M')
df['Year'] = df['Exit Time'].dt.to_period('Y')

# === CALCULATE RETURN PERCENT ===
df['Return_Pct'] = df['PnL'] / capital_per_trade * 100

# === DAILY % RETURN & SHARPE ===
daily_return_pct = df.groupby('Date')['Return_Pct'].sum()
mean_daily = daily_return_pct.mean()
std_daily = daily_return_pct.std()
sharpe = (mean_daily / std_daily) * (365 ** 0.5)

# === PLOT CUMULATIVE PNL ===
daily_pnl = df.groupby('Date')['PnL'].sum()
cumulative_pnl = daily_pnl.cumsum()
cumulative_pnl.plot(title="Cumulative PnL ($)", figsize=(10, 4))
plt.tight_layout()
plt.savefig("cumulative_pnl.png")
plt.close()

# === PLOT DRAWDOWN CURVE (IN %) ===
peak = cumulative_pnl.cummax()
drawdown_pct = ((peak - cumulative_pnl) / 300000) * 100
drawdown_pct.plot(title="Drawdown (%)", figsize=(10, 4), color='red')
plt.tight_layout()
plt.savefig("drawdown_curve.png")
plt.close()

# === MONTHLY, YEARLY, TICKER RETURNS IN % ===
monthly_returns = df.groupby('Month')['Return_Pct'].sum()
yearly_returns = df.groupby('Year')['Return_Pct'].sum()
returns_by_ticker = df.groupby('Ticker')['Return_Pct'].sum()

monthly_returns.plot(kind='bar', title="Monthly Returns (%)", figsize=(12, 4))
plt.tight_layout()
plt.savefig("monthly_returns.png")
plt.close()

yearly_returns.plot(kind='bar', title="Yearly Returns (%)", figsize=(8, 4))
plt.tight_layout()
plt.savefig("yearly_returns.png")
plt.close()

returns_by_ticker.plot(kind='bar', title="Returns by Ticker (%)", figsize=(10, 4))
plt.tight_layout()
plt.savefig("returns_by_ticker.png")
plt.close()

# === TRADE METRICS ===
total_trades = len(df)
winning_trades = df[df['PnL'] > 0]
win_rate = len(winning_trades) / total_trades

tp_count = (df['Exit Reason'] == 'Take Profit').sum()
sl_count = (df['Exit Reason'] == 'Stop Loss').sum()

gross_pnl = df['PnL'].sum()

# Calculate total fees and slippage in absolute terms for net pnl
total_fee = capital_per_trade * fee_perc * 2 * total_trades  # round trip fee
total_slip = capital_per_trade * slip_perc * 2 * total_trades  # round trip slippage

net_pnl = gross_pnl - total_fee - total_slip

# === PDF REPORT ===
class PDF(FPDF):
    def header(self):
        self.set_font("Arial", 'B', 16)
        self.cell(0, 10, "Strategy Prospectus", 0, 1, 'C')

    def add_section(self, title, content):
        self.set_font("Arial", 'B', 12)
        self.cell(0, 10, title, 0, 1)
        self.set_font("Arial", '', 11)
        self.multi_cell(0, 8, content)
        self.ln()

    def add_image(self, img_path, title=None):
        if title:
            self.set_font("Arial", 'B', 12)
            self.cell(0, 10, title, 0, 1)
        self.image(img_path, w=180)
        self.ln(5)

pdf = PDF()
pdf.add_page()

# === SUMMARY SECTION ===
summary = f"""
Backtest Period: {daily_return_pct.index.min()} to {daily_return_pct.index.max()}
Capital Per Trade: ${capital_per_trade:,.0f}
Total Trades: {total_trades}
Win Rate: {win_rate * 100:.2f}%
Take Profit Trades: {tp_count}
Stop Loss Trades: {sl_count}
Sharpe Ratio (based on daily % returns): {sharpe:.2f}
Gross PnL ($): ${gross_pnl:,.2f}
Estimated Fees: {fee_perc * 100:.3f}% each side
Estimated Slippage: {slip_perc * 100:.3f}% each side
Take Profit (TP): {tp_perc:.2f}%
Stop Loss (SL): {sl_perc:.2f}%
Max Drawdown (%): {drawdown_pct.max():.2f}%
"""

pdf.add_section("Performance Summary", summary)

# === ADD CHARTS ===
pdf.add_image("cumulative_pnl.png", "Cumulative PnL ($)")
pdf.add_image("drawdown_curve.png", "Drawdown (%)")
pdf.add_image("monthly_returns.png", "Monthly Returns (%)")
pdf.add_image("yearly_returns.png", "Yearly Returns (%)")
pdf.add_image("returns_by_ticker.png", "Returns by Ticker (%)")

# === SAVE PDF ===
pdf.output("strategy_prospectus.pdf")
print("📄 Saved: strategy_prospectus.pdf")
